Gaussien kernel

Simulation and estimators calculations

In [ ]:
import numpy as np
from MHP import MHP

mu = np.array([0.3, 0.2], dtype=float)

alpha = np.array([
    [0.30, 0.10],
    [0.20, 0.25]
], dtype=float)


def gaussian_pdf(t, m, s):
    t = np.asarray(t, dtype=float)
    return np.exp(-0.5 * ((t - m) / s) ** 2) / (s * np.sqrt(2.0 * np.pi))

def make_bimodal_kernel(a, m1, s1, m2, s2, w1=0.5, n_sigmas=5.0):
    cutoff = max(m1 + n_sigmas * s1, m2 + n_sigmas * s2)

    def phi(t):
        t_arr = np.asarray(t, dtype=float)
        vals = a * (
            w1 * gaussian_pdf(t_arr, m1, s1)
            + (1.0 - w1) * gaussian_pdf(t_arr, m2, s2)
        )
        vals = np.where((t_arr >= 0.0) & (t_arr <= cutoff), vals, 0.0)

        if np.ndim(t) == 0:
            return float(vals)
        return vals

    phi.cutoff = cutoff
    return phi


Phi = np.empty((2, 2), dtype=object)

Phi[0, 0] = make_bimodal_kernel(alpha[0, 0], m1=0.12, s1=0.035, m2=0.55, s2=0.08,  w1=0.55)
Phi[0, 1] = make_bimodal_kernel(alpha[0, 1], m1=0.15, s1=0.040, m2=0.65, s2=0.10,  w1=0.60)
Phi[1, 0] = make_bimodal_kernel(alpha[1, 0], m1=0.10, s1=0.035, m2=0.50, s2=0.085, w1=0.50)
Phi[1, 1] = make_bimodal_kernel(alpha[1, 1], m1=0.14, s1=0.040, m2=0.60, s2=0.09,  w1=0.55)

mhp = MHP(Phi=Phi, mu=mu, T_max=1.5)

print("rho =", mhp.check_stability())
print("K =\n", mhp.kernel_norm_matrix())
print("Lambda =", mhp.stationary_rate())

mhp.plot_kernels(t_max=1.5)

data = mhp.generate(horizon=3000.0, seed=123)
print(data[:10])
print("n_events =", len(data))

mhp.plot_intensity(data, horizon=10.0)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from data import EventData
from StatisticEstimators import *
from MHP import MHP

D = 2
Lambda_true = np.linalg.solve(np.eye(D) - alpha, mu)

# Simulate
horizon = 10000.0
hp = MHP(Phi=Phi, mu=mu, T_max=1.5)
hp.check_stability()
raw = hp.generate(horizon=horizon)
print(f"Simulated {len(raw):,} events  (T={horizon:.0f})")

events = EventData(
    times=raw[:, 0],
    types=raw[:, 1].astype(int),
    marks=np.ones(len(raw)),   # dummy, unused here
    horizon=horizon,
)

# Estimate statistics
Lambda_hat = estimate_Lambda(events)
lag_max, h = 1.0, 0.15
G_hat, t_grid, _ = estimate_G(
    events,
    Lambda_hat,
    lag_max=lag_max,
    h=h,
    n_lin=30,
    n_log=80,
)

print(f"True  λ : {np.round(Lambda_true, 4)}")
print(f"Est.  λ : {np.round(Lambda_hat,  4)}")
print(f"Error   : {np.round(np.abs(Lambda_hat - Lambda_true), 5)}")


def true_G_from_kernel(Phi, t_grid):
    """
    Solve
        G(t) = Phi(t) + ∫_0^t Phi(s) G(t-s) ds
    on a grid by causal quadrature.

    Parameters
    ----------
    Phi : (D,D) object array of callables
        Kernel matrix. Phi[i,j](t) must accept scalar/array t >= 0.
    t_grid : (n,) array
        Increasing positive time grid.

    Returns
    -------
    G : (n,D,D) array
        True second-order statistic on t_grid.
    """
    t_grid = np.asarray(t_grid, dtype=float)
    n = len(t_grid)
    if n < 2:
        raise ValueError("Need at least 2 time points.")

    D = Phi.shape[0]
    G = np.zeros((n, D, D), dtype=float)

    dt = np.diff(np.concatenate(([0.0], t_grid)))

    Phi_vals = np.zeros((n, D, D), dtype=float)
    for k, t in enumerate(t_grid):
        for i in range(D):
            for j in range(D):
                Phi_vals[k, i, j] = float(Phi[i, j](t))

    for n_idx in range(n):
        G[n_idx] = Phi_vals[n_idx].copy()
        for m in range(n_idx):
            k = n_idx - m - 1
            G[n_idx] += Phi_vals[m] @ G[k] * dt[m]

    return G


G_true = true_G_from_kernel(Phi, t_grid)

# Plot
CT, CE = "#1C3F6E", "#D95F02"
label = {
    (0, 0): "self  1→1",
    (0, 1): "cross 2→1",
    (1, 0): "cross 1→2",
    (1, 1): "self  2→2",
}

fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
fig.suptitle(
    rf"$\hat G$ vs True $G$"
    f"\nT={horizon:.0f},  N={len(raw):,}",
    fontsize=13,
    fontweight="bold",
)

for i in range(D):
    for j in range(D):
        ax = axes[i, j]
        tv = G_true[:, i, j]
        ev = G_hat[i, j, :]

        rl2_raw = (
            np.sqrt(np.trapezoid((tv - ev) ** 2, t_grid))
            / (np.sqrt(np.trapezoid(tv ** 2, t_grid)) + 1e-12)
        ) * 100

        ax.plot(
            t_grid,
            ev,
            color=CE,
            lw=1.4,
            alpha=0.9,
            label=rf"$\hat G^{{{i+1}{j+1}}}$  (L2={rl2_raw:.0f}%)",
        )
        ax.plot(
            t_grid,
            tv,
            color=CT,
            lw=2.5,
            label=rf"True $G^{{{i+1}{j+1}}}$",
        )
        ax.axhline(0, color="gray", lw=0.7, ls=":")

        ax.set_title(rf"$G^{{{i+1}{j+1}}}(t)$ — {label[(i, j)]}", fontsize=11)
        ax.set_ylabel(rf"$G^{{{i+1}{j+1}}}(t)$")
        if i == 1:
            ax.set_xlabel("lag $t$")
        ax.legend(fontsize=8.5, loc="upper right")
        ax.grid(ls="--", alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

Model Training and results

In [ ]:
import torch
import copy
import numpy as np
import matplotlib.pyplot as plt
from models import KernelRowNet
from StatisticEstimators import build_H
from loss import temporal_weights

device = torch.device("cpu")

# Hyperparameters
HIDDEN      = 64
LAYERS      = 1
LR0         = 1e-3
Q           = 250
BATCH_SIZE  = 8
TRAIN_SIZE  = 1024
VAL_SIZE    = 128
EPOCHS      = 500
EPS_W       = 5.0
PRINT_EVERY = 100

t_min = t_grid[0]
T     = lag_max

# Fixed quadrature grid
e1 = np.logspace(np.log10(t_min), np.log10(0.05),  80 + 1)
e2 = np.logspace(np.log10(0.05),  np.log10(0.3),   100 + 1)
e3 = np.logspace(np.log10(0.3),   np.log10(T),      70 + 1)
edges = np.unique(np.concatenate([e1, e2[1:], e3[1:]]))
t_np  = 0.5 * (edges[:-1] + edges[1:])
w_np  = np.diff(edges)

# edges = np.logspace(np.log10(t_min), np.log10(T), Q + 1)
# t_np  = 0.5 * (edges[:-1] + edges[1:])
# w_np  = np.diff(edges)

t_th = torch.tensor(t_np, dtype=torch.float32, device=device)
w_th = torch.tensor(w_np, dtype=torch.float32, device=device)

# Random collocation grids
rng        = np.random.default_rng(42)
t_train_np = np.sort(rng.uniform(t_min, T, TRAIN_SIZE))
t_val_np   = np.sort(rng.uniform(t_min, T, VAL_SIZE))

# Build H from raw G_hat, not smoothed G
H_func = build_H(G_hat, t_grid, Lambda_hat)

def compute_H_coll(t_coll, t_quad, H_func, D):
    N, Qq = len(t_coll), len(t_quad)
    diff  = t_coll[:, None] - t_quad[None, :]   # (N, Q)
    H     = np.zeros((N, Qq, D, D), dtype=float)
    for k in range(D):
        for j in range(D):
            H[:, :, k, j] = H_func(k, j, diff.ravel()).reshape(N, Qq)
    return torch.tensor(H, dtype=torch.float32, device=device)

print("Precomputing H for training collocation points")
H_train_th = compute_H_coll(t_train_np, t_np, H_func, D)

print("Precomputing H for validation collocation points")
H_val_th   = compute_H_coll(t_val_np, t_np, H_func, D)

# Targets: raw G_hat interpolated at collocation points
def interp_G_target(t_coll):
    G_interp = np.zeros((D, D, len(t_coll)), dtype=float)
    for i in range(D):
        for j in range(D):
            G_interp[i, j, :] = np.interp(t_coll, t_grid, G_hat[i, j, :])
    return torch.tensor(G_interp, dtype=torch.float32, device=device)

G_train_th = interp_G_target(t_train_np)   # (D, D, TRAIN_SIZE)
G_val_th   = interp_G_target(t_val_np)     # (D, D, VAL_SIZE)

# Log-time inputs
log_t_train = torch.log10(torch.tensor(t_train_np, dtype=torch.float32, device=device).clamp_min(1e-8))
x_train     = torch.ones_like(log_t_train)

log_t_val   = torch.log10(torch.tensor(t_val_np, dtype=torch.float32, device=device).clamp_min(1e-8))
x_val       = torch.ones_like(log_t_val)

log_t_quad  = torch.log10(t_th.clamp_min(1e-8))
x_quad      = torch.ones_like(log_t_quad)

def train_row_dgm(row_i):
    model = KernelRowNet(
        input_dim=2,
        hidden_dim=HIDDEN,
        output_dim=D,
        n_layers=LAYERS
    ).to(device)

    opt   = torch.optim.Adam(model.parameters(), lr=LR0, weight_decay=1e-6)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS, eta_min=1e-6)

    target_train = G_train_th[row_i].T   # (TRAIN_SIZE, D)
    target_val   = G_val_th[row_i].T     # (VAL_SIZE, D)

    best_val, best_epoch, best_state = float("inf"), -1, None
    patience, no_improve = 120, 0
    steps_per_epoch = max(1, TRAIN_SIZE // BATCH_SIZE)

    for ep in range(EPOCHS):

        #compute residual-based temporal weights on full train grid
        model.eval()
        with torch.no_grad():
            phi_all  = model(log_t_train, x_train)   # (TRAIN_SIZE, D)
            phi_q_ng = model(log_t_quad, x_quad)     # (Qquad, D)

            int_all = torch.einsum(
                "qk,nqkj->nj",
                phi_q_ng * w_th[:, None],
                H_train_th
            )  # (TRAIN_SIZE, D)

            res_all = phi_all + int_all - target_train
            w_all   = temporal_weights(res_all, eps=EPS_W)

        #minibatch optimization
        model.train()
        perm       = torch.randperm(TRAIN_SIZE, device=device)
        epoch_loss = 0.0

        for step in range(steps_per_epoch):
            idx = perm[step * BATCH_SIZE : (step + 1) * BATCH_SIZE]
            if idx.numel() == 0:
                continue

            H_b   = H_train_th[idx]     # (BATCH_SIZE, Qquad, D, D)
            tgt_b = target_train[idx]   # (BATCH_SIZE, D)
            w_b   = w_all[idx]          # (BATCH_SIZE, D)

            phi_b    = model(log_t_train[idx], x_train[idx])   # (BATCH_SIZE, D)
            phi_quad = model(log_t_quad, x_quad)               # (Qquad, D)

            integral = torch.einsum(
                "qk,nqkj->nj",
                phi_quad * w_th[:, None],
                H_b
            )

            residual = phi_b + integral - tgt_b
            loss = (w_b * residual.square()).mean()

            opt.zero_grad()
            loss.backward()
            opt.step()

            epoch_loss += loss.item()

        sched.step()

        # Validation
        model.eval()
        with torch.no_grad():
            phi_v  = model(log_t_val, x_val)
            phi_qv = model(log_t_quad, x_quad)

            int_v = torch.einsum(
                "qk,nqkj->nj",
                phi_qv * w_th[:, None],
                H_val_th
            )

            val_loss = (phi_v + int_v - target_val).square().mean().item()

        if val_loss < best_val:
            best_val, best_epoch = val_loss, ep
            best_state = copy.deepcopy(model.state_dict())
            no_improve = 0
        else:
            no_improve += 1

        if ep % PRINT_EVERY == 0 or ep == EPOCHS - 1:
            print(
                f"[row {row_i}] ep {ep:04d}  "
                f"lr={opt.param_groups[0]['lr']:.3e}  "
                f"train={epoch_loss / steps_per_epoch:.4e}  "
                f"val={val_loss:.4e}  "
                f"best={best_val:.4e}@ep{best_epoch}"
            )


    if best_state is not None:
        model.load_state_dict(best_state)

    print(f"-> loaded best checkpoint (ep {best_epoch}, val={best_val:.4e})")
    return model

trained_models = []
for row_i in range(D):
    print(f"\n{'='*60}\nTraining row {row_i}\n{'='*60}")
    trained_models.append(train_row_dgm(row_i))

# Predict on fine grid
t_fine   = np.logspace(np.log10(t_np[0]), np.log10(t_np[-1]), 300)
log_fine = torch.log10(torch.tensor(t_fine, dtype=torch.float32, device=device).clamp_min(1e-8))
x_fine   = torch.ones_like(log_fine)

phi_pred      = np.zeros((D, D, len(t_fine)))
phi_true_fine = np.zeros_like(phi_pred)

for i in range(D):
    trained_models[i].eval()
    with torch.no_grad():
        phi_pred[i] = trained_models[i](log_fine, x_fine).cpu().numpy().T

# True kernel from custom callable Phi
for i in range(D):
    for j in range(D):
        phi_true_fine[i, j, :] = np.asarray(Phi[i, j](t_fine), dtype=float)

# Plot
fig, axes = plt.subplots(2, 2, figsize=(11, 8), sharex=True)
fig.suptitle(
    r"NN kernel $\hat\phi$ vs True $\phi$"
    f"\n(neurons={HIDDEN}, depth={LAYERS}, batch={BATCH_SIZE}, "
    f"train={TRAIN_SIZE}, val={VAL_SIZE}, epochs={EPOCHS})",
    fontsize=13,
    fontweight="bold",
)

lbl = {
    (0, 0): "self 1->1",
    (0, 1): "cross 2->1",
    (1, 0): "cross 1->2",
    (1, 1): "self 2->2",
}

for i in range(D):
    for j in range(D):
        ax = axes[i, j]
        pt = phi_true_fine[i, j]
        pp = phi_pred[i, j]

        rl2 = (
            np.sqrt(np.trapezoid((pt - pp) ** 2, t_fine))
            / (np.sqrt(np.trapezoid(pt ** 2, t_fine)) + 1e-12)
        ) * 100

        ax.plot(t_fine, pt, color="#1C3F6E", lw=2.5, label=rf"True $\phi_{{{i+1}{j+1}}}$")
        ax.plot(
            t_fine,
            pp,
            color="#D95F02",
            lw=2.0,
            ls="--",
            label=rf"NN $\phi_{{{i+1}{j+1}}}$ (L2={rl2:.0f}%)",
        )
        ax.axhline(0, color="gray", lw=0.7, ls=":")
        ax.set_xscale("log")
        ax.set_title(rf"$\phi_{{{i+1}{j+1}}}(t)$ -- {lbl[(i, j)]}", fontsize=11)
        ax.set_ylabel(rf"$\phi_{{{i+1}{j+1}}}(t)$")
        if i == 1:
            ax.set_xlabel("lag $t$")
        ax.legend(fontsize=9)
        ax.grid(ls="--", alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()

In [ ]:
from WH import WienerHopf

# Wiener-Hopf estimation on the same grid
wh_model = WienerHopf(n_fft=65536, tikhonov=1e-4).fit(G_hat, t_grid)

phi_wh = wh_model.predict(t_fine)   # (D, D, len(t_fine))

K_wh = wh_model.kernel_norms()

#comparison plot
lbl = {(0,0): "self 1->1", (0,1): "cross 2->1",
       (1,0): "cross 1->2", (1,1): "self 2->2"}

fig, axes = plt.subplots(2, 2, figsize=(12, 8), sharex=True)
fig.suptitle(
    r"Kernel comparison — True $\phi$ vs Neural Hawkes vs Wiener-Hopf"
    f"\n(T={horizon:.0f}, N={len(raw):,}, alpha=[[0.30,0.10],[0.20,0.25]]",
    fontsize=13, fontweight="bold",
)

C_TRUE = "#1C3F6E"
C_NN   = "#D95F02"
C_WH   = "#2CA02C"

for i in range(D):
    for j in range(D):
        ax = axes[i, j]
        pt = phi_true_fine[i, j]
        pp = phi_pred[i, j]
        pw = phi_wh[i, j]

        rl2_nn = (np.sqrt(np.trapezoid((pt - pp)**2, t_fine))
                  / (np.sqrt(np.trapezoid(pt**2, t_fine)) + 1e-12)) * 100
        rl2_wh = (np.sqrt(np.trapezoid((pt - pw)**2, t_fine))
                  / (np.sqrt(np.trapezoid(pt**2, t_fine)) + 1e-12)) * 100

        ax.plot(t_fine, pt, color=C_TRUE, lw=2.5,
                label=rf"True $\phi^{{{i+1}{j+1}}}$")
        ax.plot(t_fine, pp, color=C_NN, lw=2.0, ls="--",
                label=rf"NN  (rL2={rl2_nn:.0f}%)")
        ax.plot(t_fine, pw, color=C_WH, lw=2.0, ls="-.",
                label=rf"WH  (rL2={rl2_wh:.0f}%)")

        ax.axhline(0, color="gray", lw=0.7, ls=":")
        ax.set_xscale("log")
        ax.set_title(rf"$\phi^{{{i+1}{j+1}}}(t)$ — {lbl[(i,j)]}", fontsize=11)
        ax.set_ylabel(rf"$\phi^{{{i+1}{j+1}}}(t)$")
        if i == 1:
            ax.set_xlabel("lag  $")
        ax.legend(fontsize=9, loc="upper right")
        ax.grid(ls="--", alpha=0.3)
        ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.show()
